# Reproduce the detection ablation

Reproduces the two tables of the detection ablation (tile level on the test set, mosaic level on the Diciembre
orthomosaics) from the published checkpoints: it evaluates the 16 detectors (8 architectures x 2 training sets)
with their fixed, validation-selected score thresholds (`ablations/thresholds.json`), builds the LaTeX tables
and compares every value with the published ones.

Runs on **Kaggle** or **Google Colab**, in about 15–20 minutes:
- *Kaggle*: *Add Input* → `paulauzcategui/mixed-resolution-frailejones` and
  `paulauzcategui/low-resolution-frailejones-detection`; *Settings* → *Accelerator: GPU T4* and *Internet on*.
- *Colab*: *Runtime → Change runtime type → T4 GPU*; the datasets are downloaded with your Kaggle API token.

The notebook keeps the platform's PyTorch and installs only the missing detection packages. With a different
PyTorch / CUDA build a detection right at a threshold can flip, so the check allows a tolerance of 0.005 on the
metrics and reports how many values are identical. Section 6 (optional, slow) installs the exact environment
of the published results (`requirements.txt`, Python 3.12) and checks for an exact match.

## 1. Code

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/kaggle/working" if IN_KAGGLE else "/content"
REPO = f"{WORK}/Mapping-rosettes" if (IN_COLAB or IN_KAGGLE) else os.getcwd()
if (IN_COLAB or IN_KAGGLE) and not os.path.exists(REPO):
    !git clone --branch reproduce https://github.com/pauzca/Mapping-rosettes.git {REPO}
%cd {REPO}

## 2. Packages

Only the detection packages, at the versions used for the results; the platform's torch, torchvision and numpy
are kept as they are (pip constraint).

In [ ]:
if IN_COLAB or IN_KAGGLE or os.environ.get("ROSETTES_INSTALL"):
    !pip freeze | grep -iE "^(torch|torchvision|numpy)==" > /tmp/keep.txt
    !pip install -q -c /tmp/keep.txt ultralytics==8.3.238 deepforest==2.1.0 transformers==5.12.1 \
        "geodataset @ git+https://github.com/hugobaudchon/geodataset.git@bf5034e01226219999828a81b5d03395730312e5"
!python -c "import torch, ultralytics, transformers, deepforest, geodataset; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__, '| transformers', transformers.__version__)"

## 3. Data

On Kaggle the attached datasets are used directly. On Colab they are downloaded: add your Kaggle API token as a
Colab secret named `KAGGLE_API_TOKEN` (key icon on the left) or upload your `kaggle.json` when asked.

In [ ]:
import glob
DATASETS = {"mixed": "mixed-resolution-frailejones", "lowres": "low-resolution-frailejones-detection"}

def find_dataset(slug):
    # Kaggle mounts attached datasets under /kaggle/input (the exact depth varies)
    hits = sorted(glob.glob(f"/kaggle/input/**/{slug}/dataset/data.yaml", recursive=True))
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None

if IN_KAGGLE:
    ROOTS = {r: find_dataset(slug) for r, slug in DATASETS.items()}
    assert all(ROOTS.values()), f"attach both datasets as notebook inputs: {ROOTS}"
else:
    DATA = os.environ.get("ROSETTES_DATA", "/content/data")
    if not all(os.path.exists(f"{DATA}/{slug}/dataset/data.yaml") for slug in DATASETS.values()):
        if IN_COLAB:
            from google.colab import userdata
            try:
                os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
            except Exception:
                from google.colab import files
                print("Upload kaggle.json"); files.upload()
                os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
                os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
                os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        for slug in DATASETS.values():
            !kaggle datasets download paulauzcategui/{slug} -p {DATA}/{slug} --unzip -q
    ROOTS = {r: f"{DATA}/{slug}" for r, slug in DATASETS.items()}
print(ROOTS)

## 4. Evaluate the 16 detectors

Each model is evaluated on the test tiles (per source) and on the two Diciembre orthomosaics, at its fixed
validation-selected threshold. Results go to `outputs/eval/<regime>/<model>/`.

In [ ]:
import json
THRESHOLDS = json.load(open("ablations/thresholds.json"))
ULTRALYTICS = {"yolo11s", "yolov8s", "yolo11m", "rtdetr_l"}
MOSAICS = f"{ROOTS['mixed']}/mosaics"

def evaluate_all(out_root):
    for regime, models in THRESHOLDS.items():
        for model, thr in models.items():
            ckpt = f"{ROOTS[regime]}/checkpoints/{model}"
            spec = f"ultralytics:{ckpt}/weights/best.pt" if model in ULTRALYTICS else f"run:{ckpt}"
            print(f"=== {regime} / {model} (threshold {thr})", flush=True)
            !python -m ablations.evaluate_model --model {spec} --thr {thr} --data {ROOTS[regime]}/dataset/data.yaml \
                --mosaics-root {MOSAICS} --out {out_root}/{regime}/{model} 2>&1 | grep -E "^(test all|dic)|Error"

evaluate_all("outputs/eval")

## 5. Tables and check against the published values

In [ ]:
!python -m ablations.tables --eval-root outputs/eval
!python -m ablations.check --eval-root outputs/eval --tol 0.005

In [ ]:
from IPython.display import Markdown, display
for name in ("tabla_teselas", "tabla_mosaicos"):
    display(Markdown(f"**{name}.tex**\n```latex\n{open(f'outputs/eval/{name}.tex').read()}\n```"))

## 6. (Optional, slow) Exact environment

Installs the full environment of the published results (`requirements.txt`: Python 3.12, PyTorch 2.9.1 built
for CUDA 12.6) and re-evaluates; every count and metric must then match exactly. Takes considerably longer
and needs Python 3.12 (Colab); set `RUN_EXACT = True` to run it.

In [ ]:
RUN_EXACT = False
if RUN_EXACT:
    !pip install -q -r requirements.txt
    for regime, models in THRESHOLDS.items():
        for model, thr in models.items():
            ckpt = f"{ROOTS[regime]}/checkpoints/{model}"
            spec = f"ultralytics:{ckpt}/weights/best.pt" if model in ULTRALYTICS else f"run:{ckpt}"
            !python -m ablations.evaluate_model --model {spec} --thr {thr} --data {ROOTS[regime]}/dataset/data.yaml \
                --mosaics-root {MOSAICS} --out outputs/eval_exact/{regime}/{model} 2>&1 | grep -E "Error"
    !python -m ablations.check --eval-root outputs/eval_exact --tol 0